In [ ]:
import numpy as np
import pandas as pd

In [ ]:
f_techno = snakemake.input.technoeconomic_database
f_scen = snakemake.input.scenario_db
zones = pd.read_csv(snakemake.input.zones).loc[:, "zone"]

psys_scen = snakemake.wildcards.psys_scenario

out_ledger = snakemake.output.ledger

In [ ]:
planning_horizon = int(snakemake.params.planning_horizon)

In [ ]:
planning_horizon

In [ ]:
scen = pd.read_excel(f_scen, sheet_name="scenario_tech_definition", skiprows=0)
scen = scen.loc[(scen["Psys Scenario"] == psys_scen), :]
techs = scen["Technology Name (highRES)"]

In [ ]:
gen_lifetime = (
    pd.read_excel(f_techno, sheet_name="gen", skiprows=1, engine="calamine")
    .loc[:, ["Technology Name (highRES)", "lifetime"]]
    .rename(columns={"Technology Name (highRES)": "Technology"})
)

In [ ]:
store_lifetime = (
    pd.read_excel(f_techno, sheet_name="store", skiprows=1, engine="calamine")
    .loc[:, ["Technology Name (highRES)", "e lifetime", "p lifetime"]]
    .rename(columns={"Technology Name (highRES)": "Technology"})
    .melt(id_vars="Technology", var_name="capacity_type", value_name="lifetime")
)

In [ ]:
store_lifetime["capacity_type"] = store_lifetime["capacity_type"].map(
    {"e lifetime": "ecap", "p lifetime": "pcap"}
)

In [ ]:
def melt_zone_wide(df, zones, value_name):
    m = df.melt(
        id_vars=["Technology", "parameter"],
        value_vars=list(zones),
        var_name="zone",
        value_name=value_name,
    )
    return m.dropna(subset=[value_name])

In [ ]:
def rledger(fin, techs, zones, Y0):
    lim = pd.read_excel(fin, sheet_name="gen_exist_r", skiprows=0, engine="calamine")
    lim = lim.loc[
        (lim["Year"] == Y0)
        & (lim["Technology"].isin(techs))
        & (lim["zone"].isin(zones)),
        :,
    ].rename(columns={"value": "capacity_mw", "Installed year": "installed_year"})

    lim["capacity_type"] = "pcap"
    lim["planning_horizon"] = Y0

    return lim[
        ["Technology", "zone", "region", "capacity_type", "capacity_mw", "installed_year", "planning_horizon"]
    ]

In [ ]:
def zledger(fin, sheet, techs, zones, tech_type, Y0):
    lim = pd.read_excel(fin, sheet_name=sheet, skiprows=0, engine="calamine")
    lim = lim.loc[(lim["Year"] == Y0) & (lim["Technology"].isin(techs)), :]

    cap = lim.loc[
        lim["parameter"].isin([tech_type + "_exist_pcap_z", tech_type + "_exist_ecap_z"]), :
    ]
    vintage = lim.loc[lim["parameter"] == tech_type + "_installed_year_z", :]

    cap_long = melt_zone_wide(cap, zones, "capacity_mw")
    cap_long["capacity_type"] = cap_long["parameter"].str.extract(r"_(pcap|ecap)_z")

    vintage_long = melt_zone_wide(vintage, zones, "installed_year")

    out = cap_long.merge(
        vintage_long[["Technology", "zone", "installed_year"]],
        on=["Technology", "zone"],
        how="left",
    )
    out["region"] = np.nan
    out["planning_horizon"] = Y0

    return out[
        ["Technology", "zone", "region", "capacity_type", "capacity_mw", "installed_year", "planning_horizon"]
    ]


In [ ]:
r_gen = rledger(f_techno, techs, zones, planning_horizon)

In [ ]:
z_only_gen = techs[techs.isin(["HydroRes", "NuclearEPR"])]
z_gen = zledger(f_techno, "gen_exist_z", z_only_gen, zones, "gen", planning_horizon)

z_only_store = techs[techs.isin(["PumpedHydro"])]
z_store = zledger(f_techno, "store_exist_z", z_only_store, zones, "store", planning_horizon)

In [ ]:
ledger = pd.concat([z_gen, r_gen, z_store], ignore_index=True)

ledger = ledger.merge(gen_lifetime, on="Technology", how="left", suffixes=("", "_gen"))
ledger = ledger.merge(
    store_lifetime, on=["Technology", "capacity_type"], how="left", suffixes=("", "_store")
)
ledger["lifetime"] = ledger["lifetime"].fillna(ledger["lifetime_store"])
ledger = ledger.drop(columns="lifetime_store")

In [ ]:
ledger = ledger.sort_values(["Technology", "zone", "region"]).reset_index(drop=True)
ledger["installed_year"] = ledger["installed_year"].astype("Int64")

In [ ]:
ledger.to_csv(out_ledger, index=False)

In [ ]:
ledger